# 关于模型调用的invoke的使用

## 1. invoke的传参

### 1.1 文本输入

举例：

In [9]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

# 加载配置文件
load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GOOGLE_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# 確保環境變數有 GOOGLE_API_KEY，或是直接在這裡把 key 傳進 api_key 參數
model = init_chat_model(
    model="gemma-4-31b-it",
    model_provider="openai",  # 關鍵：強制使用 OpenAI 的連線邏輯
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/" # 指向 Google
)
print(model)

metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.2.12'}} profile={} client=<openai.resources.chat.completions.completions.Completions object at 0x000001AC1C54D940> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001AC1C54DE80> root_client=<openai.OpenAI object at 0x000001AC1B0B2CF0> root_async_client=<openai.AsyncOpenAI object at 0x000001AC1C54DBE0> model_name='gemma-4-31b-it' model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='https://generativelanguage.googleapis.com/v1beta/openai/'


In [10]:
response = model.invoke("翻譯如下中文: 你好世界")
print(response)

content='<thought>*   Source text: "你好世界" (Chinese)\n    *   Target language: English (implied, as it\'s the most common request for translation unless specified otherwise, but providing a few common options is better).\n    *   Content: A classic greeting often used in programming ("Hello World").\n\n    *   "你好" (Nǐ hǎo) $\\rightarrow$ Hello / Hi\n    *   "世界" (Shìjiè) $\\rightarrow$ World\n\n    *   Direct translation: "Hello World"\n    *   Informal: "Hi World"\n\n    *   The phrase "你好世界" is the standard translation of the programming phrase "Hello, World!".\n\n    *   English: Hello World</thought>英文翻譯為：**Hello World**' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 8, 'total_tokens': 169, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemma-4-31b-it', 'system_fingerprint': None, 'id': 'mbduavmrAZ2i9tMPmZfH8QE', 'finish_reason': 'stop', 'logprobs': None}

## 2. invoke的返回值

举例1：

In [13]:
from langchain_core.messages import SystemMessage, HumanMessage,AIMessage
response = model.invoke([HumanMessage(content="2 + 3 * 2 = ?")])

In [14]:
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


In [15]:
print(response)

content='<thought>*   Expression: `2 + 3 * 2`\n    *   Goal: Calculate the result.\n\n    *   Multiplication and division are performed before addition and subtraction.\n    *   The operation sequence is:\n        1.  Multiplication: `3 * 2`\n        2.  Addition: `2 + (result of multiplication)`\n\n    *   Step 1: $3 \\times 2 = 6$\n    *   Step 2: $2 + 6 = 8$\n\n    *   Result is 8.</thought>2 + 3 * 2 = **8**' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 10, 'total_tokens': 141, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemma-4-31b-it', 'system_fingerprint': None, 'id': 'B7huasb4CNKi9tMPgJ2E8AE', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fc07e-deee-7700-801d-cf7a8205e19b-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 10, 'output_tokens': 11, 'total_tokens': 141, 'input_token_details': {}, 'output_token_details'

美化输出：

In [16]:
from rich import print as rprint

rprint(response)

AIMessage(
    content='<thought>*   Expression: `2 + 3 * 2`\n    *   Goal: Calculate the result.\n\n    *   Multiplication 
and division are performed before addition and subtraction.\n    *   The operation sequence is:\n        1.  
Multiplication: `3 * 2`\n        2.  Addition: `2 + (result of multiplication)`\n\n    *   Step 1: $3 \\times 2 = 
6$\n    *   Step 2: $2 + 6 = 8$\n\n    *   Result is 8.</thought>2 + 3 * 2 = **8**',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 11,
            'prompt_tokens': 10,
            'total_tokens': 141,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_provider': 'openai',
        'model_name': 'gemma-4-31b-it',
        'system_fingerprint': None,
        'id': 'B7huasb4CNKi9tMPgJ2E8AE',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019fc07e-deee-7700-801d-cf7a8205e19b-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 10,
        'output_tokens': 11,
        'total_tokens': 141,
        'input_token_details': {},
        'output_token_details': {}
    }
)

举例2：

In [13]:
response = model.invoke("用一句话解释什么是 AI")

# 1. 获取回复内容
print("AI 回复:", response.content)

# 2. 获取响应元数据
metadata = response.response_metadata
print(f"使用的模型: {metadata['model_name']}")
print(f"结束原因: {metadata['finish_reason']}")
print(f"模型提供商：{metadata['model_provider']}\n")

# 3. 获取 Token 使用情况
usage = metadata.get('token_usage', {})
print(f"输入 tokens: {usage.get('prompt_tokens')}")
print(f"输出 tokens: {usage.get('completion_tokens')}")
print(f"总计 tokens: {usage.get('total_tokens')}")

# 4. 获取消息 ID
print(f"消息 ID: {response.id}")

AI 回复: AI（人工智能）是让计算机像人一样学习、思考和完成任务的技术。
使用的模型: gpt-5.4-mini-2026-03-17
结束原因: stop
模型提供商：openai

输入 tokens: 13
输出 tokens: 26
总计 tokens: 39
消息 ID: lc_run--019e4081-87e8-7d83-907c-86591fdf8de9-0
